# Discretization calibration {#sec-calibration}

In this section, we run error tests to determine which reward sensitivity $\sigma$ to choose in each environment. These results also give us insights into the effect of discretization on strategy. We run these tests on our three main environments:
* two-armed bandit task;
* risk-reward dilemma;
* stag-hunt game;

First, we import what we need.

In [ ]:
# general imports

import numpy as np

import jax.numpy as jnp

import matplotlib.pyplot as plt
import matplotlib as mpl

import pandas as pd
import seaborn as sns

In [ ]:
# pyCRLD imports

# Agents
from pyCRLD.Agents.Base import abase
from pyCRLD.Agents.StrategySARSA import stratSARSA
from pyCRLD.Agents.StrategyActorCritic import stratAC

# Environments
from pyDDRL.Environments.SocialDilemma import SocialDilemma
from pyDDRL.Environments.RiskRewardDilemma import RiskRewardDilemma
from pyDDRL.Environments.TwoArmedBandit import TwoArmedBandit

In [ ]:
# DDRL imports

from pyDDRL.Agents.DDRLBase import DDRLBase
from pyDDRL.Agents.DDRLSarsa import DDRLSarsa
from pyDDRL.Agents.DDRLActorCritic import DDRLActorCritic

## Errors on a two-armed bandit task

Here, we compare a distributional and a non-distributional SARSA agent playing a "rich" two-armed bandit task ($p_1 = 0.9$, $p_2 = 0.7$) and a "poor" two-armed bandit task ($p_1 = 0.3$, $p_2 = 0.1$). The error at $X$ is defined as the norm of the difference between the temporal difference error of a distributional agent and that of a non-distributional agent for a given strategy $X$. We compute the error for different strategies $X$, and plot the mean error as a function of reward sensitivity $\sigma$. In @sec-results , we will select the reward sensitivity that minimizes the error in order to compute our results.

First, we define a "poor" and a "rich" environment:

In [ ]:
# Poor environment

xs = np.array([[-1, 1],
               [-1, 1]])
ps = np.array([[0.7, 0.3],
               [0.9, 0.1]])
env_poor = TwoArmedBandit(disttype='discrete', xs=xs, ps=ps)

In [ ]:
# Rich environment

xs = np.array([[-1, 1],
               [-1, 1]])
ps = np.array([[0.1, 0.9],
               [0.3, 0.7]])
env_rich = TwoArmedBandit(disttype='discrete', xs=xs, ps=ps)

Now we define the reward sensitivities that we are going to compare, as well as the strategies for which we will compute the error:

In [ ]:
# List of sigmas to compare
sigmas = np.linspace(0.0001, 1., 40)

# Sample strategies
Xs = np.linspace(0.01, 0.99, 11)

Here is the function that will allow us to compute the error:

In [ ]:
# Function to compute mean error

def compute_mean_error(env, Xs, sigmas, path, version):
    errors = np.zeros((len(sigmas), len(Xs)))
    for j, sigma in enumerate(sigmas):  # loop over sigmas
        print("sigma = "+str(sigma))
        # non-distributional agent
        mae = stratSARSA(env=env, learning_rates=0.1, 
                         discount_factors=0, choice_intensities=4., use_prefactor=True)
        
        # distributional agent
        aei = DDRLSarsa(env, nr_reward_bins=29, discretization_sigma=sigma, 
                        learning_rates=0.1, discount_factors=0, 
                        choice_intensities=4, Rmin=-1, Rmax=1,
                        method="weights", wU=1, use_prefactor=True)
        
        for k, x in enumerate(Xs):  # loop over strategies
            # strategy X
            X = np.zeros((1, 1, 2))
            X[:, :, 0] = x
            X[:, :, 1] = 1 - x

            # TDe for distributional agent
            distRPE = aei.TDerror(X, norm=True)

            # TDe for non-distributional agent
            detRPE = mae.TDerror(X, norm=True)

            # compute the error
            error = np.linalg.norm(detRPE-distRPE) 
            errors[j, k] = error
    
    # compute the mean error
    mean_error = np.mean(errors, axis=-1)

    # put results into df and save
    columns = ["X="+str(x) for x in Xs]
    index = ["sigma="+str(sigma) for sigma in sigmas]
    df = pd.DataFrame(errors, index=index, columns=columns)
    df.to_csv(path+"tab_error_df_"+version+".csv")
    return mean_error

In [ ]:
# Path and version
path = "../data/error_calculation/"
version_poor = "poor_v1"
version_rich = "rich_v1"

In [ ]:
#| eval: false
# Compute mean error for poor and rich envs
mean_err_poor = compute_mean_error(env_poor, Xs, sigmas, path, version_poor)
mean_err_rich = compute_mean_error(env_rich, Xs, sigmas, path, version_rich)

In [ ]:
# Extract the data
df_poor = pd.read_csv(path+"tab_error_df_"+version_poor+".csv")
df_rich = pd.read_csv(path+"tab_error_df_"+version_rich+".csv")

# Convert to np array
mean_err_poor = df_poor.to_numpy()[:, 1:]
mean_err_rich = df_rich.to_numpy()[:, 1:]

Now let's plot the error:

In [ ]:
#| label: fig-error-tab
#| fig-cap: "Mean error in a poor and in a rich environment as a function of reward sensitivity."

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
ax1.plot(sigmas, mean_err_poor)
ax2.plot(sigmas, mean_err_rich)
ax1.set_xlabel("discretization "+r'$\sigma$')
ax2.set_xlabel("discretization "+r'$\sigma$')
ax1.set_title("mean error in a poor environment")
ax2.set_title("mean error in a rich environment")
ax1.set_ylabel("mean error")
plt.show()

<Figure size 2400x900 with 2 Axes>

The reward sensitivity that minimizes the error is:

In [ ]:
idx_rich, idx_poor = np.argmin(mean_err_rich), np.argmin(mean_err_poor)
sigmas[idx_rich], sigmas[idx_poor]

(np.float64(0.0001), np.float64(0.0001))

## Errors in a stag-hunt game

Now we compare a distributional with a non-distributional agent playing the stag-hunt game defined @sec-socialdilemma , with $R = 3$, $T = 1$, $P = 0$ and $S = -2$. 

In [ ]:
env = SocialDilemma(R=3, P=0, S=-2, T=1)

Once again, we compare different reward sensitivities $\sigma$ for the distributional agent. Since the game is symmetric in agents, we compute the error assuming that the second agent's strategy equals the first agent's strategy.

In [ ]:
# List of sigmas to compare
sigmas = np.linspace(0.0001, 1., 40)

# Sample strategies
Xs = np.linspace(0.01, 0.99, 11)

In [ ]:
# Function to compute mean error
def compute_mean_error(env, Xs, sigmas, gamma, path, version):
    errors = np.zeros((len(sigmas), len(Xs)))
    for j, sigma in enumerate(sigmas):  # loop over sigmas
        print("sigma = "+str(sigma))
        # Non-distributional agent
        mae = stratSARSA(env=env, learning_rates=0.1, choice_intensities=50,
                         discount_factors=gamma, use_prefactor=True)
        
        # Distributional agent
        aei = DDRLSarsa(env, nr_reward_bins=29, discretization_sigma=sigma, 
                        learning_rates=0.1, discount_factors=gamma, 
                        Rmin=-2, Rmax=3, choice_intensities=50,
                        method="weights", wU=1, use_prefactor=True)
        
        for k, x in enumerate(Xs):  # loop over strategies
            X = np.zeros((2, 1, 2))
            X[0, 0, 0] = x
            X[0, 0, 1] = 1 - x
            # Both agents have the same strategy
            X[1, 0, 0] = x
            X[1, 0, 1] = 1 - x
            
            # TDe for dist agent
            distRPE = aei.TDerror(X, norm=True)

            # TDe for non-dist agent
            detRPE = mae.TDerror(X, norm=True)

            # Compute the error
            error = np.linalg.norm(detRPE-distRPE) 
            errors[j, k] = error

    # Compute mean error        
    mean_error = np.mean(errors, axis=-1)

    # Format into df and save
    columns = ["X="+str(x) for x in Xs]
    index = ["sigma="+str(sigma) for sigma in sigmas]
    df = pd.DataFrame(errors, index=index, columns=columns)
    df.to_csv(path+"sh_error_df_"+version+".csv")
    return mean_error

In [ ]:
# path and version
path = "../data/error_calculation/"
version = "v0"

We compute the error for a SARSA agent with discount factor $\gamma = 0.9$ and choice intensity $\beta = 50$, as in @sec-results .

In [ ]:
#| eval: false
# Compute mean error
mean_err = compute_mean_error(env, Xs, sigmas, 0.9, path, version)

In [ ]:
# Extract the data
df_error = pd.read_csv(path+"sh_error_df_"+version+".csv")

# Convert to np array
mean_err = np.mean(df_error.to_numpy()[:, 1:], axis=1)

Now we can plot the error as a function of $\sigma$:

In [ ]:
#| label: fig-error-sh
#| fig-cap: "Mean error in a stag-hunt game as a function of reward sensitivity."

# Plot
fig, ax1 = plt.subplots(1, 1, figsize=(4, 3))
ax1.plot(sigmas, mean_err)
ax1.set_xlabel("discretization "+r'$\sigma$')
ax1.set_title("mean error for "+r'$\gamma=0.9$')
ax1.set_ylabel("mean error")
plt.show()

<Figure size 1200x900 with 1 Axes>

The reward sensitivity that minimizes the error is:

In [ ]:
idx = np.argmin(mean_err)
sigmas[idx]

np.float64(0.07701538461538461)

## Errors in a risk-reward dilemma

Here, we compute the error for a distributional Actor-Critic agent playing a risk-reward dilemma as defined in @sec-rrdilemma , with $p_c = 0.2$, $p_r = 0.1$, $r_c = 0.5$, $r_r = 1$ and $r_d = 0$. 

In [ ]:
env = RiskRewardDilemma(pc=0.2, pr=0.1, rc=0.5, rr=1, rd=0)

Since the risk-reward dilemma contains 2 states, the discount factor $\gamma$ has a significant impact on the agent's strategy. Therefore, we will compute the error for an agent with a low discount factor ($\gamma=0.4$), and for an agent with a high discount factor ($\gamma=0.9$). In the degraded state, Actor-Critic agents converge to a maximally safe strategy since it is the only rational one (see @sec-results ). Therefore, we compute the error for different strategies in the prosperous state, assuming the strategy in the degraded state is always $X^{safe} = 1$.

In [ ]:
# Discount factors
gammas = [0.4, 0.9]

# List of sigmas to compare
sigmas = np.linspace(0.0001, 1., 40)

# Sample strategies
Xs = np.linspace(0.01, 0.99, 11)

In [ ]:
# Function to compute mean error
def compute_mean_error(env, Xs, sigmas, gamma, path, version):
    errors = np.zeros((len(sigmas), len(Xs)))
    for j, sigma in enumerate(sigmas):  # loop over sigmas
        print("sigma = "+str(sigma))
        # Non-distributional agent
        mae = stratAC(env=env, learning_rates=0.1, 
                      discount_factors=gamma, use_prefactor=True)
        
        # Distributional agent
        aei = DDRLActorCritic(env, nr_reward_bins=29, discretization_sigma=sigma,
                              learning_rates=0.1, discount_factors=gamma, 
                              Rmin=0, Rmax=1,
                              method="weights", wU=1, use_prefactor=True)
        
        for k, x in enumerate(Xs):  # Loop over strategies
            X = np.zeros((1, 2, 2))
            X[:, 0, 0] = x
            X[:, 0, 1] = 1 - x
            X[:, 1, 0] = 1  # strategy in deg state

            # TDe for dist agent
            distRPE = aei.TDerror(X, norm=True)

            # TDe for non-dist agent
            detRPE = mae.TDerror(X, norm=True)

            # Compute error
            error = np.linalg.norm(detRPE-distRPE) 
            errors[j, k] = error

    # Compute mean error        
    mean_error = np.mean(errors, axis=-1)

    # Format into df and save
    columns = ["X="+str(x) for x in Xs]
    index = ["sigma="+str(sigma) for sigma in sigmas]
    df = pd.DataFrame(errors, index=index, columns=columns)
    df.to_csv(path+"rr_error_df_"+version+".csv")
    return mean_error

In [ ]:
# Path and version
path = "../data/error_calculation/"
version_low = "low_v1"
version_high = "high_v1"

In [ ]:
#| eval: false
# Mean error for low gamma
mean_err_low = compute_mean_error(env, Xs, sigmas, gammas[0], path, version_low)

# Mean error for high gama
mean_err_high = compute_mean_error(env, Xs, sigmas, gammas[1], path, version_high)

In [ ]:
# Extract the data
df_error_low = pd.read_csv(path+"rr_error_df_"+version_low+".csv")
df_error_high = pd.read_csv(path+"rr_error_df_"+version_high+".csv")

# Convert to np array
mean_err_low = np.mean(df_error_low.to_numpy()[:, 1:], axis=1)
mean_err_high = np.mean(df_error_high.to_numpy()[:, 1:], axis=1)

Now we can plot the error for both values of $\gamma$:

In [ ]:
#| label: fig-error-rr
#| fig-cap: "Mean error in a risk-reward dilemma as a function of reward sensitivity."

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
ax1.plot(sigmas, mean_err_low)
ax2.plot(sigmas, mean_err_high)
ax1.set_xlabel("discretization "+r'$\sigma$')
ax2.set_xlabel("discretization "+r'$\sigma$')
ax1.set_title("mean error for "+r'$\gamma=0.4$')
ax2.set_title("mean error for "+r'$\gamma=0.9$')
ax1.set_ylabel("mean error")
plt.show()

<Figure size 2400x900 with 2 Axes>

The reward sensitivity that minimizes the error for low discount factor is:

In [ ]:
idx = np.argmin(mean_err_low)
sigmas[idx]

np.float64(0.05137692307692308)

And for high discount factor:

In [ ]:
idx = np.argmin(mean_err_high)
sigmas[idx]

np.float64(0.025738461538461536)